In [1]:
import numpy as np
from sklearn.linear_model import Lasso
import matplotlib.pyplot as plt

import setting
import utils
import parametric
import CoRT_builder
from joblib import Parallel, delayed 
from tqdm import tqdm
import json
import datetime
import os

def parametric_uniform_test(n_target, n_source, p, K, Ka, h, lamda, alpha, T, cnt):
    # import os # Import here to get the Process ID
    # print(f"--- Worker started at {datetime.datetime.now()} ---", flush=True)

    s_vector = [0] * cnt
    s = len(s_vector)
    CoRT_model = CoRT_builder.CoRT(alpha=lamda)

    target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s, "AR")
    similar_source_index = CoRT_model.find_similar_source(n_target, K, target_data, source_data, T=T, verbose=False)
    X_combined, y_combined = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data)

    model = Lasso(alpha=lamda, **setting.lasso_config)
    model.fit(X_combined, y_combined.ravel())
    beta_hat_target = model.coef_[-p:]

    active_indices = np.array([i for i, b in enumerate(beta_hat_target) if b != 0])

    if len(active_indices) == 0:
        print(f"Iteration {iter}: Lasso selected no features. Skipping.")

    j = np.random.choice(len(active_indices))

    X_target = target_data["X"]
    y_target = target_data["y"]
    X_active, X_inactive = utils.get_active_X(beta_hat_target, X_target)

    etaj, etajTy = utils.construct_test_statistic(y_target, j, X_active)

    Sigma = np.eye(n_target)
    b_global = Sigma @ etaj @ np.linalg.pinv(etaj.T @ Sigma @ etaj)
    a_global = (Sigma - b_global @ etaj.T) @ y_target

    folds = utils.split_target(T, X_target, y_target, n_target)

    tn_sigma = (np.sqrt(etaj.T @ Sigma @ etaj)).item()
    z_k = -20 * tn_sigma
    z_max = 20 * tn_sigma
    
    Z_train_list = parametric.get_Z_train(z_k, folds, source_data, a_global, b_global, lamda, K, T)
    Z_val_list, similar_source_index, cnt_vote, is_similar = parametric.get_Z_val(z_k, folds, T, K, a_global, b_global, lamda, source_data)

    target_data_current = {"X": X_target, "y": a_global + z_k * b_global}
    X_combined_new, y_combined_new = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data_current)
    L_CoRT, R_CoRT, Az = parametric.get_Z_CoRT(X_combined_new, similar_source_index, lamda, a_global, b_global, source_data, z_k)
    
    z_list = [z_k]
    Az_list = []
    stopper = "empty"

    while z_k < z_max:
        print(f"zk:{z_k}")
        current_num_sources = len(similar_source_index)
        offset = p * current_num_sources
        Az_target_current = np.array([idx - offset for idx in Az if idx >= offset])
        Az_list.append(Az_target_current)
        mn = z_max
        stopper = "MAX"

        for val in Z_train_list:
            if mn > val[4]:
                mn = val[4]
                stopper = "TRAIN"

        for val in Z_val_list:
            if mn > val[3]:
                mn = val[3]
                stopper = "VAL"

        if mn > R_CoRT:
            mn = R_CoRT
            stopper = "CORT"

        R_final = mn

        z_k = max(R_final, z_k) + 1e-8

        if (z_k >= z_max):
            z_list.append(z_max)
            break
        else:
            z_list.append(z_k)

        update_train_needed = False
        update_val_needed = False
        update_cort_needed = False
        
        if stopper == "TRAIN":
            update_train_needed = True
            update_val_needed = True   

        elif stopper == "VAL":
            update_val_needed = True
            
        elif stopper == "CORT":
            update_cort_needed = True

        if update_train_needed:
            for val in Z_train_list:
                if val[4] <= z_k + 1e-9:
                    l, r = parametric.update_Z_train(val, z_k, folds, source_data, a_global, b_global, lamda, K, T)
                    val[3] = l
                    val[4] = r

        if update_val_needed:
            for val in Z_val_list:
                if stopper == "TRAIN" or val[3] <= z_k + 1e-9:
                    l, r, similar_source_index, cnt_vote, is_similar, oke = parametric.update_Z_val(val, z_k, folds, T, a_global, b_global, lamda, source_data, similar_source_index, cnt_vote, is_similar)
                    val[2] = l
                    val[3] = r
                    if oke == True:
                        update_cort_needed = True

        if update_cort_needed:
            target_data_current = {"X": X_target, "y": a_global + z_k * b_global}
            X_combined_new, y_combined_new = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data_current)
            L_CoRT, R_CoRT, Az = parametric.get_Z_CoRT(X_combined_new, similar_source_index, lamda, a_global, b_global, source_data, z_k)

    optim_p_value = utils.pivot(active_indices, Az_list, z_list, etaj, etajTy, 0, Sigma)

    if optim_p_value == 0:
        print(" WARNING: p-value is 0")
    elif optim_p_value is None:
        print(" WARNING: p-value is None")

    # print(f"--- Worker {os.getpid()} FINISHED at {datetime.datetime.now().strftime('%H:%M:%S')} ---")
    return optim_p_value

In [10]:
n_target = 50
n_source = 20
p = 50
K = 5 
Ka = 3
h = 30 
lamda = 0.05
alpha = 0.05
iteration = 1
T = 5
cnt = 5
owner = "HiepDepTrai"
p_values = []

print("Start simulation")

for i in range(iteration):
    p_value = parametric_uniform_test(n_target, n_source, p, K, Ka, h, lamda, alpha, T, cnt)
    p_values.append(p_value)

results = {
    "timestamp": datetime.datetime.now().isoformat(),
    "owner": owner,
    "params": {
        "n_target": n_target,
        "n_source": n_source,
        "p": p, 
        "K": K,
        "Ka": Ka,
        "h": h, 
        "lamda": lamda,
        "alpha": alpha,
        "iteration": iteration,
        "T": T,
        "cnt": cnt
    },
    "p_values": p_values,
    "len": len(p_values)
}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

output_filename = f"p-{p}_uniform_test.json"

if not os.path.exists(output_filename):
    all_results = [results]
else: 
    with open(output_filename, "r", encoding="utf-8") as f:
        all_results = json.load(f)
        all_results.append(results)

total_p_values = 0
for r in all_results:
    total_p_values += r["len"]
print(f"Total p-value: {total_p_values}")

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(all_results, f, cls=NumpyEncoder, indent=4)

print(f"Đã lưu kết quả thành công vào file: {output_filename}")

Start simulation
zk:-3.6418357315679306
zk:-3.6347193231876584
zk:-3.6347193131876585
zk:-3.62622798624435
zk:-3.6199349066801654
zk:-3.6170234894536293
zk:-3.6019034303665185
zk:-3.6019034203665186
zk:-3.6019034103665186
zk:-3.6019034003665187
zk:-3.600802099173951
zk:-3.5885104127027385
zk:-3.584942920923433
zk:-3.5815851709985256
zk:-3.5700025994797637
zk:-3.5612946494304105
zk:-3.5557139940691114
zk:-3.5451032131348934
zk:-3.541712135037386
zk:-3.5359262356258867
zk:-3.5355596238966953
zk:-3.5355596138966954
zk:-3.531736937593344
zk:-3.5131886819797096
zk:-3.504167783192991
zk:-3.4770149922337037
zk:-3.474392223796892
zk:-3.4736279537540784
zk:-3.470972208019064
zk:-3.4703480542863048
zk:-3.470348044286305
zk:-3.463499081484568
zk:-3.4479239178505425
zk:-3.444249586550227
zk:-3.4413562817278147
zk:-3.440859400608603
zk:-3.43261540040981
zk:-3.4307642182946054
zk:-3.4283730259752145
zk:-3.4229961295070908
zk:-3.4229888498780148
zk:-3.420328961075651
zk:-3.4156802722560835
zk:-3.3900